# 参数与扫描

这是独立的合成沙盒，不连接设备。无需完成其他课程。先从新 kernel 顺序运行，再做文末的小修改。
启动入口已准备环境与服务；重复打开继续当前练习，重置会创建新的起点。

In [ ]:
import sys
from pathlib import Path

import scopecat as sc

project = sc.open_project()
if Path(sys.prefix).resolve() != (project.root / ".venv").resolve():
    raise RuntimeError(f"请在 Select Kernel 中选择 {project.root / '.venv'}")

In [ ]:
import numpy as np

from lab_teaching.session import open_parameters

session = project.authoring()
session.refresh()
from my_experiment.teaching import teaching_rabi

params = open_parameters(session)

参数来自独立的合成起点。修改工作区、构造扫描和预览本身都不采集。

In [ ]:
from lab_teaching.parameters import Drive

params[Drive]["q0"].frequency = 5.145
saved = params.save(note="参数沙盒")
request = teaching_rabi().sweep(amplitude=np.linspace(0, 0.8, 7))
prepared = session.prepare(request, parameters=params)
print("预览点数:", prepared.preview.point_count)
assert prepared.preview.point_count == 7

运行后，每个扫描点保存 64 个 shot。下面同时显示复数均值，原始 shot 仍保留。

In [ ]:
run = prepared.run().wait(timeout=120).result()
shots = np.asarray(run.measurements()["iq"].require_values())
assert shots.shape == (7, 64)
print("run:", run.id)
print("每个扫描点的平均 IQ:", shots.mean(axis=1))

小修改：把扫描改为 5 点，重新预览和运行，观察形状。上面的 7 点断言也要随预期修改。
关闭 Notebook 前可运行任务“停止实验服务”；再次使用沙盒入口会重新启动。